Load data mentah dan siapkan coding katagori

In [3]:
import json
from pathlib import Path

BASE_DIR = Path("./sources")

CATEGORIES = {
    "politik": "1_news-politik/3_politik_cnn_articles.json",
    "ekonomi": "2_news-ekonomi/3_ekonomi_cnn_articles.json",
    "teknologi": "3_news-teknologi/3_teknologi_cnn_articles.json",
    "olahraga": "4_news-olahraga/3_olahraga_cnn_articles.json",
    "hiburan": "5_news-hiburan/3_hiburan_cnn_articles.json",
}

LABEL_MAP = {cat: idx for idx, cat in enumerate(CATEGORIES.keys())}
# {'politik': 0, 'ekonomi': 1, 'teknologi': 2, 'olahraga': 3, 'hiburan': 4}

Konversi artikel mentah menjadi format semi terstruktur

```json
{
  "title": "Kata Istana soal Ketua Ombudsman Hery Susanto Dipecat Tidak Hormat",
  "text": "--\n\nMenteri Sekretaris Negara Prasetyo Hadi menyatakan pihaknya menghormati keputusan Majelis Etik Ombudsman RI (ORI) ... nikel tahun 2013-2025.\n\n\n\nsource on Google Addas a preferredsource on Google",
  "label": 0,
  "category": "politik"
}
```

In [4]:
def load_all_articles():
    """Load semua artikel dari semua kategori, return list of dict."""
    all_articles = []

    for category, rel_path in CATEGORIES.items():
        full_path = BASE_DIR / rel_path
        with open(full_path, "r", encoding="utf-8") as f:
            articles = json.load(f)

        for article in articles:
            all_articles.append(
                {
                    "title": article.get("title", ""),
                    "text": article.get("text", ""),
                    "label": LABEL_MAP[category],
                    "category": category,
                }
            )

        print(f"  Loaded {len(articles):>5} artikel dari [{category}]")

    print(f"\n  Total artikel: {len(all_articles)}")
    return all_articles

In [5]:
raw_data = load_all_articles()
print(raw_data[0])

  Loaded  2032 artikel dari [politik]
  Loaded  2028 artikel dari [ekonomi]
  Loaded  2027 artikel dari [teknologi]
  Loaded  2029 artikel dari [olahraga]
  Loaded  2029 artikel dari [hiburan]

  Total artikel: 10145
{'title': 'Kata Istana soal Ketua Ombudsman Hery Susanto Dipecat Tidak Hormat', 'text': '--\n\nMenteri Sekretaris Negara Prasetyo Hadi menyatakan pihaknya menghormati keputusan Majelis Etik Ombudsman RI (ORI) yang menjatuhkan sanksi pemberhentian tidak dengan hormat (PTDH) terhadap Ketua Ombudsman RI nonaktif Hery Susanto yang tersangkut kasus dugaan korupsi.\n\n"Ya, kita menghormati keputusan itu ya," kata Prasetyo di Kompleks Istana Kepresidenan, Jakarta, Senin (8/6).\n\nPrasetyo mengatakan pihaknya akan menindaklanjuti keputusan tersebut sesuai ketentuan yang berlaku.\n\nADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT\n\n"Nanti kita tindaklanjuti semuanya," ucapnya.\n\nDia menegaskan peristiwa yang berujung pada sanksi pemberhentian tersebut tidak diharapkan terjadi pada 

Proses cleaning raw text dan standarisasi sumber data

### Daftar Noise yang Harus Dibersihkan

| # | Noise | Prioritas | Contoh dalam Data |
|---|---|---|---|
| 1 | Prefix `--` di awal teks | 🔴 Wajib | `"--\n\nPenasihat Khusus..."` |
| 2 | Teks iklan `ADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT` | 🔴 Wajib | Muncul di tengah hampir setiap artikel |
| 3 | Footer scraper `source on Google Addas a preferredsource on Google` | 🔴 Wajib | Muncul di akhir setiap artikel |
| 4 | Embed multimedia `[Gambas:Video CNN]`, `[Gambas:Instagram]`, dll. | 🟡 Penting | Non-teks content |
| 5 | Navigasi halaman `Lanjut ke sebelah...` | 🟡 Penting | Artikel multi-halaman (Hiburan) |
| 6 | Blok `Pilihan Redaksi` + daftar artikel rekomendasi | 🟡 Penting | Link internal CNN |
| 7 | Caption foto (`Foto: ...`, `(dok. ...)`) | 🟡 Penting | Keterangan gambar |
| 8 | Karakter `X` tunggal berdiri sendiri | 🟢 Opsional | Sisa tombol close iklan |
| 9 | Newline berlebihan → normalisasi ke spasi | 🟡 Penting | Pemisah paragraf scraper |
| 10 | Whitespace berlebihan → normalisasi | 🟡 Penting | Sisa setelah cleaning |

### Contoh Sebelum vs Sesudah Cleaning

**Sebelum:**
```
--

Penasihat Khusus Presiden Bidang Ketenagakerjaan dan Kesejahteraan Buruh 
Said Iqbal menyatakan siap melaporkan...

ADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT

"Tapi kami bisa meyakinkan kalau menteri enggak bekerja..."

...membangun Indonesia yang lebih sejahtera," katanya.



source on Google Addas a preferredsource on Google
```

**Sesudah (text):**
```
Said Iqbal Mengaku Bakal Adukan Menteri Tak Pro Buruh ke Prabowo. Penasihat Khusus Presiden Bidang Ketenagakerjaan dan Kesejahteraan Buruh Said Iqbal menyatakan siap melaporkan... "Tapi kami bisa meyakinkan kalau menteri enggak bekerja..." ...membangun Indonesia yang lebih sejahtera," katanya.

In [6]:
import re

def clean_text(text: str) -> str:
    """
    Membersihkan teks berita dari berbagai noise hasil scraping.
    Urutan eksekusi penting — jangan diubah sembarangan.
    """

    # 1. Hapus prefix '--' di awal teks
    text = re.sub(r"^--\s*", "", text)

    # 2. Hapus teks iklan
    text = text.replace("ADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT", "")

    # 3. Hapus footer scraper
    text = text.replace("source on Google Addas a preferredsource on Google", "")

    # 4. Hapus embed multimedia [Gambas:*]
    text = re.sub(r"\[Gambas:\w+\]", "", text)

    # 5. Hapus navigasi halaman
    text = text.replace("Lanjut ke sebelah...", "")

    # 6. Hapus blok "Pilihan Redaksi" dan daftar artikel setelahnya
    text = re.sub(r"Pilihan Redaksi.*?(?=\n[A-Z]|\Z)", "", text, flags=re.DOTALL)

    # 7. Hapus caption foto
    text = re.sub(r"Foto:\s*\(.*?\)", "", text)  # Foto: (Tangkapan layar ...)
    text = re.sub(r"Foto:.*?Foto:.*?\n?", "", text)  # Foto duplikat
    text = re.sub(r"\(dok\.\s*[^)]+\)", "", text)  # (dok. Nama Fotografer)

    # 8. Hapus karakter 'X' tunggal yang berdiri sendiri (sisa tombol close)
    text = re.sub(r"(?<=\s)X(?=\s)", "", text)

    # 9. Normalisasi newline → spasi tunggal
    #    PENTING: ganti \n dengan spasi, bukan hapus, agar kata tidak menempel
    text = re.sub(r"\n+", " ", text)

    # 10. Normalisasi whitespace berlebihan → spasi tunggal
    text = re.sub(r"\s+", " ", text)

    # 11. Strip leading/trailing whitespace
    text = text.strip()

    return text

In [7]:
cleaned_data = []
for article in raw_data:
    cleaned_article = article.copy()
    cleaned_article['text'] = clean_text(article['text'])
    cleaned_data.append(cleaned_article)

print(f"Total Artikel yang dibersihkan: {len(cleaned_data)}")
print("Contoh teks bersih:\n", cleaned_data[0]['text'][:200] if cleaned_data else "")

Total Artikel yang dibersihkan: 10145
Contoh teks bersih:
 Menteri Sekretaris Negara Prasetyo Hadi menyatakan pihaknya menghormati keputusan Majelis Etik Ombudsman RI (ORI) yang menjatuhkan sanksi pemberhentian tidak dengan hormat (PTDH) terhadap Ketua Ombuds


Penggabungan title (Judul Berita) dengan teks berita

- Karena judul berita adalah sinyal klasifikasi yang sangat kuat.
- Judul adalah **ringkasan paling padat** dari isi berita. Ditulis jurnalis untuk menangkap esensi artikel dalam satu kalimat.
- Karena title ditempatkan di **awal teks gabungan**, ia dijamin **tidak akan terpotong** oleh truncation 512 token.
- Format input menjadi: `"Judul Berita. Isi berita lengkap..."`. Model membaca judul dulu, langsung paham konteksnya.

In [8]:
def combine_title_and_text(title: str, text: str) -> str:
    """
    Gabungkan title dan text menjadi satu string.
    Title ditempatkan di awal, diakhiri titik, diikuti isi berita.
    
    Contoh output:
      "Said Iqbal Mengaku Bakal Adukan Menteri Tak Pro Buruh ke Prabowo. Penasihat Khusus..."
    """
    title = title.strip()
    text = text.strip()
    
    # Pastikan title diakhiri titik sebagai pemisah
    if title and not title.endswith("."):
        title += "."
    
    return f"{title} {text}".strip()

In [9]:
for article in cleaned_data:
    article['text'] = combine_title_and_text(article['title'], article['text'])

print("Sample teks gabungan:\n", cleaned_data[0]['text'][:200] if cleaned_data else "")

Sample teks gabungan:
 Kata Istana soal Ketua Ombudsman Hery Susanto Dipecat Tidak Hormat. Menteri Sekretaris Negara Prasetyo Hadi menyatakan pihaknya menghormati keputusan Majelis Etik Ombudsman RI (ORI) yang menjatuhkan s


Setelah semua langkah cleaning telah dilakukan, data teks diexport kembali.

Untuk selanjutnya digunakan dalam proses training

In [10]:
import os
from collections import Counter

def validate_and_export(data: list, output_path: str):
    """
    Validasi data yang sudah di-clean, cetak ringkasan, lalu export ke JSON.
    Data diasumsikan sudah melalui clean_text() dan combine_title_and_text().
    """

    # --- Ringkasan data ---
    print(f"\n{'='*50}")
    print(f"LAPORAN EXPORT")
    print(f"{'='*50}")
    print(f"  Total artikel : {len(data)}")

    # Distribusi per kategori
    dist = Counter(d["category"] for d in data)
    print(f"\n  Distribusi per kategori:")
    for cat, count in sorted(dist.items(), key=lambda x: x[1], reverse=True):
        print(f"    {cat:>12}: {count:>5} artikel")

    # Statistik panjang teks
    lengths = [len(d["text"].split()) for d in data]
    print(f"\n  Panjang teks (kata):")
    print(f"    Rata-rata : {sum(lengths)/len(lengths):.0f}")
    print(f"    Min       : {min(lengths)}")
    print(f"    Max       : {max(lengths)}")

    # Contoh hasil
    print(f"\n  Contoh hasil:")
    print(f"    \"{data[0]['text'][:120]}...\"")

    # --- Export ---
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\n  ✅ Data disimpan ke: {output_path}")
    print(f"{'='*50}")

In [11]:
OUTPUT_PATH = "./data/cleaned_articles.json"
validate_and_export(cleaned_data, OUTPUT_PATH)


LAPORAN EXPORT
  Total artikel : 10145

  Distribusi per kategori:
         politik:  2032 artikel
        olahraga:  2029 artikel
         hiburan:  2029 artikel
         ekonomi:  2028 artikel
       teknologi:  2027 artikel

  Panjang teks (kata):
    Rata-rata : 366
    Min       : 22
    Max       : 4692

  Contoh hasil:
    "Kata Istana soal Ketua Ombudsman Hery Susanto Dipecat Tidak Hormat. Menteri Sekretaris Negara Prasetyo Hadi menyatakan p..."

  ✅ Data disimpan ke: ./data/cleaned_articles.json


### Format Output `cleaned_articles.json`

```json
[
  {
    "text": "Said Iqbal Mengaku Bakal Adukan Menteri Tak Pro Buruh ke Prabowo. Penasihat Khusus Presiden Bidang Ketenagakerjaan dan Kesejahteraan Buruh Said Iqbal menyatakan siap melaporkan kepada Presiden RI Prabowo Subianto apabila menemukan menteri yang dinilai tak bekerja...",
    "label": 0,
    "category": "politik"
  },
  {
    "text": "RUU Polri: Usia Pensiun Bintara-Tamtama 59 Tahun, Perwira 60 Tahun. DPR dan pemerintah sepakat terkait perubahan batas usia pensiun anggota Polri...",
    "label": 0,
    "category": "politik"
  },
  {
    "text": "IHSG Ditutup Menguat 1,2 Persen pada Perdagangan Senin. IHSG ditutup menguat 1,2 persen pada perdagangan Senin...",
    "label": 1,
    "category": "ekonomi"
  }
]
```

> Field `text` sekarang berisi **title + body yang sudah digabung**. Title ada di awal, diikuti titik (`.`), lalu isi berita. Fase selanjutnya (split, tokenisasi, training) cukup membaca field `text` ini saja - tidak perlu penanganan khusus karena title sudah terintegrasi.